# Milestone 4 — NAS sweep on Cora (MLP / GCN / SAGE / GIN) — **Corollary 3.4** instance

Trains four small architectures on Cora (CPU + GPU), audits each against the bracket, and produces a comparative table.

**This milestone instantiates Corollary 3.4** (PA-MPC sandwich for admissible families, [`PAPER-ARXIV.md`](../../../../PAPER-ARXIV.md) §3.2):

$$
H_\mathrm{bin}^{-1}\!\big(H(Y\mid\Pi_\mathcal{A}(G, L))\big) \;\le\; \mathrm{PA\text{-}MPC}(\mathcal{A}, L) \;\le\; \tfrac{1}{2}\, H(Y\mid\Pi_\mathcal{A}(G, L)).
$$

**Admissibility (Def 3.5).** GCN, SAGE, GIN are all admissible architecture families (countable multiset aggregators, neighbour-state injectivity, etc.); **MLP is not** — it has no neighbour aggregation, so $\Pi^{(L)}_\mathrm{MLP} = \Pi^{(0)}_\mathrm{MLP}$ for all $L$. That is *why* MLP's bracket on Cora is wider and MLP loses to the structural arches — the sandwich predicts it before training.

**Julia companion (optional).** [`julia-theory/notebooks/11_aggregator_triple.jl`](../../../julia-theory/notebooks/11_aggregator_triple.jl) shows $r_T$ inflation: sum's upper bracket on Cora is $\Delta_{\max}=168\times$ looser than mean/sym.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='m4', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
from onboarding.projects.shared import (
    Partition, label_partition, wl_partition,
    hbin, hbin_inverse, upper, lower, slack, verify, bracket_of,
    sum_partition, mean_partition, max_partition,
)
print('shared modules imported')


In [ ]:
from torch_geometric.datasets import Planetoid
_CACHE = Path.home() / '.cache' / 'planetoid'
_CACHE.mkdir(parents=True, exist_ok=True)
_cora = Planetoid(root=str(_CACHE / 'Cora'), name='Cora')[0]
n = int(_cora.num_nodes)
m = int(_cora.num_edges)
K = int(_cora.y.max().item()) + 1
print(f'Cora: n={n}, m={m}, classes={K}, x.shape={tuple(_cora.x.shape)}')


In [ ]:
import torch, torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GINConv
torch.manual_seed(0); np.random.seed(0)
device = torch.device('cpu')
data = _cora.to(device)

class MLP(torch.nn.Module):
    def __init__(self, d, h, k):
        super().__init__(); self.l1=torch.nn.Linear(d,h); self.l2=torch.nn.Linear(h,k)
    def forward(self, x, ei): return self.l2(F.relu(self.l1(x)))

class GCN(torch.nn.Module):
    def __init__(self, d,h,k): super().__init__(); self.c1=GCNConv(d,h); self.c2=GCNConv(h,k)
    def forward(self,x,ei): return self.c2(F.relu(self.c1(x,ei)),ei)

class SAGE(torch.nn.Module):
    def __init__(self,d,h,k): super().__init__(); self.c1=SAGEConv(d,h); self.c2=SAGEConv(h,k)
    def forward(self,x,ei): return self.c2(F.relu(self.c1(x,ei)),ei)

class GIN(torch.nn.Module):
    def __init__(self,d,h,k):
        super().__init__()
        mlp1=torch.nn.Sequential(torch.nn.Linear(d,h),torch.nn.ReLU(),torch.nn.Linear(h,h))
        mlp2=torch.nn.Sequential(torch.nn.Linear(h,h),torch.nn.ReLU(),torch.nn.Linear(h,k))
        self.c1=GINConv(mlp1); self.c2=GINConv(mlp2)
    def forward(self,x,ei): return self.c2(F.relu(self.c1(x,ei)),ei)

def train_one(model_cls, epochs=120):
    torch.manual_seed(0)
    m = model_cls(data.num_features, 64, K)
    opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(epochs):
        m.train(); opt.zero_grad()
        out = m(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        p = m(data.x, data.edge_index).argmax(-1)
        return float((p[data.test_mask] == data.y[data.test_mask]).float().mean()), float((p != data.y).float().mean())

results = {}
for name, cls in [('MLP', MLP), ('GCN', GCN), ('SAGE', SAGE), ('GIN', GIN)]:
    acc, err = train_one(cls)
    results[name] = (acc, err)
    print(f'  {name:>5}: test_acc={acc:.3f}  full_err={err:.3f}')


In [ ]:
P = wl_partition(_cora, depth=2)
bin_e = np.array([min(e, 1-e) for e in P.e])
q = np.array(P.q)
H = float(np.sum(q * np.array([hbin(e) for e in bin_e])))
print(f'wl(L=2): H={H:.4f}, bracket=[{lower(H):.4f}, {upper(H):.4f}]')


In [ ]:
# Sanity: structural archs should beat MLP on Cora.
mlp_acc = results['MLP'][0]
gcn_acc = results['GCN'][0]
assert gcn_acc > mlp_acc - 0.02, f'M4: GCN {gcn_acc:.3f} should be ≳ MLP {mlp_acc:.3f}'
for name, (acc, err) in results.items():
    assert 0 <= err <= 1
    assert acc > 0.4, f'M4: {name} acc {acc} suspiciously low'
print(f'[GATE OK] M4: 4 arches trained; GCN={gcn_acc:.3f}, MLP={mlp_acc:.3f}, SAGE={results["SAGE"][0]:.3f}, GIN={results["GIN"][0]:.3f}')


In [ ]:
fig, ax = plt.subplots(figsize=(6,3.5))
names = list(results.keys())
accs  = [results[n][0] for n in names]
errs  = [results[n][1] for n in names]
x = np.arange(len(names))
ax.bar(x-0.18, accs, 0.36, label='test acc')
ax.bar(x+0.18, errs, 0.36, label='full err')
ax.axhline(lower(H), color='C2', ls='--', label=f'lower(H)={lower(H):.2f}')
ax.axhline(upper(H), color='C3', ls='--', label=f'upper(H)={upper(H):.2f}')
ax.set_xticks(x); ax.set_xticklabels(names); ax.legend(); ax.set_title('M4 — NAS sweep against bracket')
_plots = _PROJECTS / 'capstone' / 'milestone4' / 'plots'; _plots.mkdir(exist_ok=True)
plt.tight_layout(); fig.savefig(_plots / 'm4_nas.png', dpi=120); plt.show()


In [ ]:
reflect.log('M4.nas_sweep', f'GCN={gcn_acc:.3f} > MLP={mlp_acc:.3f}; SAGE={results["SAGE"][0]:.3f}; GIN={results["GIN"][0]:.3f}', 'HIGH')


### Modal GPU NAS sweep — the actual T4 run

All four architectures trained inside one warm T4 container.

In [ ]:
from onboarding.projects import modal_app
if not (modal_app._MODAL_OK and modal_app.app is not None):
    raise RuntimeError('Modal not installed/configured. Run `pip install modal && modal token new`.')
with modal_app.app.run():
    gpu_sweep = modal_app.nas_sweep.remote(epochs=100)
print('GPU NAS sweep:', gpu_sweep)


In [ ]:
assert isinstance(gpu_sweep, dict), f'M4-GPU: expected dict, got {type(gpu_sweep)}'
if 'device' in gpu_sweep:
    assert gpu_sweep['device'] == 'cuda', f'M4-GPU: device {gpu_sweep["device"]}'
print(f'[GATE OK] M4-GPU: nas_sweep returned keys {list(gpu_sweep.keys())}')
reflect.log('M4.modal_nas', f'T4 NAS sweep ran on Cora; keys={list(gpu_sweep.keys())}', 'HIGH')


**End of M4.**